# Notebook 14 — Multi-Object Detection: Grid Detector (YOLO-style)

## Learning Objectives
- Understand grid-based (YOLO-style) multi-object detection.
- Learn how GT boxes are assigned to grid cells by their centre coordinates.
- Train `TinyGridDetector` on images with up to 3 objects.
- Understand the objectness score and how it gates classification and box losses.
- Apply Non-Maximum Suppression (NMS) to remove duplicate detections.
- Visualise the 4×4 prediction grid and before/after NMS comparisons.

## Big Picture
Single-object detection (Notebook 13) outputs one box. But real scenes have multiple objects.
The YOLO approach: divide the image into a G×G grid. Each cell independently predicts whether
an object is there (objectness), what class it is, and where the box is. Cells fire in parallel
— there is no sequential scanning of the image.

## Dataset
**SyntheticShapes (multi_detection mode)** — 128×128 RGB images with 1–3 shapes each.
3 classes: 0=circle, 1=square, 2=triangle.

## Architecture
```
Input [B, 3, 128, 128]
  └─ CNN Backbone (same as single-object)     → [B, 128, H/4, W/4]
  └─ AdaptiveAvgPool2d(grid_size=4)           → [B, 128, 4, 4]
  └─ Conv1x1(128 → n_pred)  where n_pred=1+3+4=8
                                              → [B, 8, 4, 4]
  └─ Permute + reshape                        → [B, 4, 4, 8]
     For each grid cell [b, row, col, :]:
       [0]         = objectness logit
       [1:4]       = class logits (circle, square, triangle)
       [4:8]       = box (cx, cy, w, h) raw  → apply sigmoid for [0,1]
```

## Theory
**Grid assignment**: Each GT box is assigned to the grid cell containing its centre.
A 4×4 grid on a 128×128 image → each cell covers 32×32 pixels.

**Objectness**: Binary score: is there an object centred in this cell?
Trained with binary cross-entropy. Only cells with objects get class and box losses.

**Multi-part loss** (YOLO-style):
$$\mathcal{L} = \lambda_{obj}\mathcal{L}_{obj} + \lambda_{cls}\mathcal{L}_{cls} + \lambda_{box}\mathcal{L}_{box}$$

**NMS** (Non-Maximum Suppression): After thresholding by objectness, boxes that heavily overlap
another higher-scoring box are suppressed. Essential for removing duplicates.

## Implementation from Scratch
### 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root when running from notebooks/

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_json, save_markdown_report, update_runs_summary
from src.utils.paths import RUNS_DETECTION_DIR
from src.data.synthetic_shapes import load_shapes_data, multi_detection_collate_fn
from src.models.tiny_detector import TinyGridDetector
from src.losses.detection_losses import grid_detection_loss
from src.metrics.detection import box_cxcywh_to_xyxy, box_iou, nms
from src.visualization.detection import draw_bounding_boxes, plot_detection_examples

seed_everything(42)
device = get_best_device()
run_dir = prepare_run_dir(RUNS_DETECTION_DIR, clean=False)
print(f'Device  : {device}')
print(f'Run dir : {run_dir}')

## Dataset
### 2. Load Synthetic Shapes (multi_detection mode)

In [ ]:
IMAGE_SIZE   = 128
NUM_CLASSES  = 3   # circle, square, triangle
GRID_SIZE    = 4   # 4×4 grid
MAX_OBJECTS  = 3
BATCH_SIZE   = 16
NUM_EPOCHS   = 8
LR           = 1e-3
LAMBDA_OBJ   = 1.0
LAMBDA_CLS   = 1.0
LAMBDA_BOX   = 5.0
CLASS_NAMES  = ['circle', 'square', 'triangle']

# multi_detection_collate_fn handles variable-length object lists
train_loader, val_loader = load_shapes_data(
    n_train=800,
    n_val=200,
    image_size=IMAGE_SIZE,
    mode='multi_detection',
    max_objects=MAX_OBJECTS,
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

imgs, class_ids_list, boxes_list = next(iter(train_loader))
print(f'Image batch shape   : {imgs.shape}  (B, 3, H, W)')
print(f'class_ids_list type : {type(class_ids_list)}  (list of B lists)')
print(f'boxes_list type     : {type(boxes_list)}  (list of B tensors)')
print(f'Example classes     : {class_ids_list[0]}  (variable length per image)')
print(f'Example boxes       : {boxes_list[0].shape}  (num_objects, 4)')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

In [ ]:
# Show one sample with multiple boxes
sample_img  = imgs[0].numpy()
sample_cls  = class_ids_list[0]           # list of ints
sample_boxes = boxes_list[0].numpy()      # [N, 4] normalised cxcywh
sample_boxes_px = box_cxcywh_to_xyxy(torch.tensor(sample_boxes)).numpy() * IMAGE_SIZE

fig, ax = plt.subplots(figsize=(5, 5))
draw_bounding_boxes(
    sample_img,
    boxes=sample_boxes_px,
    class_ids=sample_cls,
    class_names=CLASS_NAMES,
    ax=ax,
)
ax.set_title(f'Sample: {len(sample_cls)} objects — {[CLASS_NAMES[c] for c in sample_cls]}')
plt.tight_layout()
fig.savefig(run_dir / 'grid_detector_sample.png', dpi=120)
plt.close(fig)
print(f'Sample with {len(sample_cls)} objects saved.')

### 3. Understand the Grid Assignment

In [ ]:
# Demonstrate grid assignment: which cell does each GT box belong to?
print(f'Grid size: {GRID_SIZE}×{GRID_SIZE}  (each cell covers {IMAGE_SIZE//GRID_SIZE}×{IMAGE_SIZE//GRID_SIZE} pixels)')
print()
for i, (cls, box) in enumerate(zip(sample_cls, sample_boxes)):
    cx, cy, w, h = box
    col = min(int(cx * GRID_SIZE), GRID_SIZE - 1)
    row = min(int(cy * GRID_SIZE), GRID_SIZE - 1)
    print(f'  Object {i}: {CLASS_NAMES[cls]:8s}  cx={cx:.3f} cy={cy:.3f} '
          f'→ assigned to grid cell (row={row}, col={col})')

### 4. Build TinyGridDetector

In [ ]:
model = TinyGridDetector(
    in_channels=3,
    num_classes=NUM_CLASSES,
    grid_size=GRID_SIZE,
).to(device)

n_params = model.count_parameters()
print(f'Trainable parameters : {n_params:,}')
print(f'Predictions per cell : {model.n_pred}  (1 obj + {NUM_CLASSES} cls + 4 box)')

# Shape trace
x_sample = imgs[:2].to(device)                    # [2, 3, 128, 128]
with torch.no_grad():
    raw = model(x_sample)                          # [2, 4, 4, 8]

print(f'Input shape  : {x_sample.shape}')
print(f'Output shape : {raw.shape}  (B, grid_size, grid_size, n_pred)')
print(f'  Objectness logits : raw[:, :, :, 0]   shape [{raw.shape[0]}, {GRID_SIZE}, {GRID_SIZE}]')
print(f'  Class logits      : raw[:, :, :, 1:4] shape [{raw.shape[0]}, {GRID_SIZE}, {GRID_SIZE}, {NUM_CLASSES}]')
print(f'  Box raw coords    : raw[:, :, :, 4:]  shape [{raw.shape[0]}, {GRID_SIZE}, {GRID_SIZE}, 4]')

## Sanity Checks

In [ ]:
assert raw.shape == (2, GRID_SIZE, GRID_SIZE, model.n_pred)
print('Output shape check passed.')

assert not torch.isnan(raw).any()
print('No NaN in output.')

# Loss computable
test_loss = grid_detection_loss(
    raw, class_ids_list[:2], [b.to(device) for b in boxes_list[:2]],
    grid_size=GRID_SIZE, num_classes=NUM_CLASSES,
    lambda_obj=LAMBDA_OBJ, lambda_cls=LAMBDA_CLS, lambda_box=LAMBDA_BOX,
    device=device,
)
print(f'Initial grid detection loss: {test_loss.item():.4f}')

# Objectness probability (random init) ~ 0.5
obj_prob = torch.sigmoid(raw[:, :, :, 0]).mean().item()
print(f'Mean objectness probability (random init): {obj_prob:.3f}  (expected ~0.5)')

print('All sanity checks passed!')

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)

history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    t_loss, n_batches = 0.0, 0
    for batch_imgs, batch_cls_list, batch_boxes_list in train_loader:
        batch_imgs = batch_imgs.to(device)                        # [B, 3, H, W]
        batch_boxes_dev = [b.to(device) for b in batch_boxes_list]
        optimizer.zero_grad()
        raw = model(batch_imgs)                                   # [B, G, G, n_pred]
        loss = grid_detection_loss(
            raw, batch_cls_list, batch_boxes_dev,
            grid_size=GRID_SIZE, num_classes=NUM_CLASSES,
            lambda_obj=LAMBDA_OBJ, lambda_cls=LAMBDA_CLS, lambda_box=LAMBDA_BOX,
            device=device,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
        n_batches += 1
    history['train_loss'].append(t_loss / n_batches)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_n = 0.0, 0
    with torch.no_grad():
        for batch_imgs, batch_cls_list, batch_boxes_list in val_loader:
            batch_imgs = batch_imgs.to(device)
            batch_boxes_dev = [b.to(device) for b in batch_boxes_list]
            raw = model(batch_imgs)
            v_loss += grid_detection_loss(
                raw, batch_cls_list, batch_boxes_dev,
                grid_size=GRID_SIZE, num_classes=NUM_CLASSES,
                lambda_obj=LAMBDA_OBJ, lambda_cls=LAMBDA_CLS, lambda_box=LAMBDA_BOX,
                device=device,
            ).item()
            v_n += 1
    history['val_loss'].append(v_loss / v_n)
    scheduler.step()

    print(f'Epoch {epoch}/{NUM_EPOCHS}  '
          f'train={history["train_loss"][-1]:.4f}  '
          f'val={history["val_loss"][-1]:.4f}')

print('Training complete!')

## Evaluation

In [ ]:
# Evaluate with decode_predictions (includes NMS)
model.eval()
total_tp, total_pred, total_gt = 0, 0, 0

with torch.no_grad():
    for batch_imgs, batch_cls_list, batch_boxes_list in val_loader:
        batch_imgs = batch_imgs.to(device)
        raw = model(batch_imgs)
        results = model.decode_predictions(raw, conf_threshold=0.4, nms_iou_threshold=0.5)

        for i, res in enumerate(results):
            pred_boxes_norm = res['boxes']     # [N, 4] cxcywh
            gt_boxes_norm   = batch_boxes_list[i]  # [M, 4] cxcywh
            total_pred += len(pred_boxes_norm)
            total_gt   += len(gt_boxes_norm)

            if len(pred_boxes_norm) == 0 or len(gt_boxes_norm) == 0:
                continue

            pred_xyxy = box_cxcywh_to_xyxy(pred_boxes_norm)
            gt_xyxy   = box_cxcywh_to_xyxy(gt_boxes_norm)
            iou_mat   = box_iou(pred_xyxy, gt_xyxy)
            matched = set()
            for pi in range(len(pred_boxes_norm)):
                for gi in range(len(gt_boxes_norm)):
                    if gi not in matched and iou_mat[pi, gi].item() >= 0.5:
                        total_tp += 1
                        matched.add(gi)
                        break

precision = total_tp / max(total_pred, 1)
recall    = total_tp / max(total_gt,   1)
f1        = 2 * precision * recall / max(precision + recall, 1e-8)

print(f'Precision @IoU0.5 : {precision:.4f}')
print(f'Recall    @IoU0.5 : {recall:.4f}')
print(f'F1 Score          : {f1:.4f}')
print(f'Total GT objects  : {total_gt}  |  Total predicted: {total_pred}  |  True positives: {total_tp}')

metrics = {
    'precision_at_iou50': float(precision),
    'recall_at_iou50': float(recall),
    'f1_score': float(f1),
    'final_train_loss': float(history['train_loss'][-1]),
    'final_val_loss': float(history['val_loss'][-1]),
    'num_params': n_params,
    'num_epochs': NUM_EPOCHS,
    'grid_size': GRID_SIZE,
}
save_metrics_json(metrics, run_dir / 'grid_detector_metrics.json')
print('Metrics saved.')

## Visualization

In [ ]:
# Training curve
from src.visualization.plots import plot_training_curves
plot_training_curves(
    history,
    save_path=run_dir / 'grid_detector_training_curve.png',
    title='TinyGridDetector Training Curves',
)
print('Training curve saved.')

In [ ]:
# Grid visualisation: show objectness scores on the 4×4 grid
model.eval()
vis_imgs, vis_cls_list, vis_boxes_list = next(iter(val_loader))
with torch.no_grad():
    vis_raw = model(vis_imgs.to(device))  # [B, 4, 4, n_pred]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # Left: image with objectness heatmap overlay
    obj_scores = torch.sigmoid(vis_raw[i, :, :, 0]).cpu().numpy()  # [4, 4]
    img_np = vis_imgs[i].numpy().transpose(1, 2, 0).clip(0, 1)     # [H, W, 3]

    axes[0][i].imshow(img_np)
    im = axes[0][i].imshow(obj_scores, alpha=0.5, cmap='hot', vmin=0, vmax=1,
                           extent=[0, IMAGE_SIZE, IMAGE_SIZE, 0])
    axes[0][i].set_title(f'Objectness grid\n(max={obj_scores.max():.2f})')
    axes[0][i].axis('off')

    # Right: decoded predictions with NMS
    res = model.decode_predictions(
        vis_raw[i:i+1], conf_threshold=0.3, nms_iou_threshold=0.5
    )[0]
    gt_boxes_px = (box_cxcywh_to_xyxy(vis_boxes_list[i]) * IMAGE_SIZE).numpy()
    pred_boxes_px = (box_cxcywh_to_xyxy(res['boxes']).cpu() * IMAGE_SIZE).numpy() if len(res['boxes']) > 0 else np.zeros((0, 4))

    axes[1][i].imshow(img_np)
    # Draw GT boxes in green
    for box in gt_boxes_px:
        x0, y0, x1, y1 = box
        rect = plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor='lime', lw=2)
        axes[1][i].add_patch(rect)
    # Draw predicted boxes in red
    for box in pred_boxes_px:
        x0, y0, x1, y1 = box
        rect = plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor='red', lw=2, linestyle='--')
        axes[1][i].add_patch(rect)
    axes[1][i].set_title(f'GT (green) vs Pred (red)\n#{len(vis_cls_list[i])} GT, #{len(res["boxes"])} pred')
    axes[1][i].axis('off')

plt.suptitle('TinyGridDetector: Objectness (top) + Predictions (bottom)', fontsize=13)
plt.tight_layout()
fig.savefig(run_dir / 'grid_detector_grid_visualisation.png', dpi=120)
plt.close(fig)
print('Grid visualisation saved.')

In [ ]:
# NMS before/after comparison on one image
model.eval()
with torch.no_grad():
    raw_single = model(vis_imgs[:1].to(device))  # [1, 4, 4, n_pred]

# Before NMS: all boxes with objectness > threshold
pred_single = raw_single[0]  # [4, 4, n_pred]
obj_scores_flat = torch.sigmoid(pred_single[:, :, 0]).view(-1)  # [16]
boxes_flat = torch.sigmoid(pred_single[:, :, 1+NUM_CLASSES:]).view(-1, 4)  # [16, 4]
cls_flat   = pred_single[:, :, 1:1+NUM_CLASSES].view(-1, NUM_CLASSES).argmax(-1)  # [16]

threshold = 0.3
keep_mask = obj_scores_flat > threshold
boxes_pre_nms   = boxes_flat[keep_mask].cpu()
scores_pre_nms  = obj_scores_flat[keep_mask].cpu()
cls_pre_nms     = cls_flat[keep_mask].cpu()

# After NMS
if len(boxes_pre_nms) > 0:
    boxes_xyxy_pre = box_cxcywh_to_xyxy(boxes_pre_nms)
    keep_nms = nms(boxes_xyxy_pre, scores_pre_nms, iou_threshold=0.5)
    boxes_post_nms = boxes_pre_nms[keep_nms]
    scores_post_nms = scores_pre_nms[keep_nms]
    cls_post_nms = cls_pre_nms[keep_nms]
else:
    boxes_post_nms = boxes_pre_nms
    cls_post_nms = cls_pre_nms

img_np = vis_imgs[0].numpy().transpose(1, 2, 0).clip(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, boxes, cls_ids, title in [
    (axes[0], boxes_pre_nms, cls_pre_nms, f'Before NMS ({len(boxes_pre_nms)} boxes)'),
    (axes[1], boxes_post_nms, cls_post_nms, f'After NMS ({len(boxes_post_nms)} boxes)'),
]:
    boxes_px = (box_cxcywh_to_xyxy(boxes) * IMAGE_SIZE).numpy() if len(boxes) > 0 else np.zeros((0, 4))
    draw_bounding_boxes(img_np, boxes_px, class_ids=cls_ids.tolist(),
                        class_names=CLASS_NAMES, ax=ax)
    ax.set_title(title)

plt.suptitle('NMS Before vs After', fontsize=12)
plt.tight_layout()
fig.savefig(run_dir / 'grid_detector_nms_comparison.png', dpi=120)
plt.close(fig)
print('NMS comparison saved.')

In [ ]:
# Final prediction examples
model.eval()
with torch.no_grad():
    full_raw = model(vis_imgs[:4].to(device))
    all_results = model.decode_predictions(full_raw, conf_threshold=0.4, nms_iou_threshold=0.5)

pred_boxes_px_list, pred_cls_list, gt_boxes_px_list, gt_cls_list = [], [], [], []
for i, res in enumerate(all_results):
    pb = (box_cxcywh_to_xyxy(res['boxes']).cpu() * IMAGE_SIZE).numpy() if len(res['boxes']) > 0 else np.zeros((0, 4))
    gb = (box_cxcywh_to_xyxy(vis_boxes_list[i]) * IMAGE_SIZE).numpy()
    pred_boxes_px_list.append(pb)
    pred_cls_list.append(res['class_ids'].cpu().tolist())
    gt_boxes_px_list.append(gb)
    gt_cls_list.append(vis_cls_list[i])

plot_detection_examples(
    images=vis_imgs[:4].numpy(),
    gt_boxes_list=gt_boxes_px_list,
    gt_class_ids_list=gt_cls_list,
    pred_boxes_list=pred_boxes_px_list,
    pred_class_ids_list=pred_cls_list,
    save_path=run_dir / 'grid_detector_predictions.png',
    title='TinyGridDetector: GT (left) vs Predicted after NMS (right)',
    n_examples=4,
    class_names=CLASS_NAMES,
)
print('Prediction examples saved.')

## Failure Cases

**Common TinyGridDetector failure patterns:**

1. **One cell — multiple objects**: If two objects are centred in the same grid cell, only one
   can be assigned. The other is lost. YOLO solves this with multiple anchors per cell.

2. **Object spans multiple cells**: Large objects with centres in one cell but extent into
   adjacent cells can confuse the localisation head.

3. **Class imbalance in cells**: Most cells predict no-object (negative). The binary cross-entropy
   for objectness is dominated by negatives. YOLO uses `lambda_noobj < lambda_obj` to downweight them.

4. **NMS threshold sensitivity**: A strict IoU threshold (e.g., 0.3) may suppress valid
   predictions of nearby but non-overlapping objects.

5. **Box regression on empty cells**: Box head predictions for no-object cells are trained only
   implicitly (they don't receive box loss). This can lead to erratic box values unless objectness
   gating is effective.

## Exercises

1. **Finer grid**: Change `grid_size` to 8. How does recall improve for small objects? How does
   training time change?

2. **No-object weight**: Add `lambda_noobj=0.5` to downweight no-object cells. Does precision
   improve?

3. **Confidence threshold sweep**: Run inference with conf_threshold ∈ {0.1, 0.3, 0.5, 0.7}.
   Plot precision and recall vs threshold. Where is the best F1?

4. **Anchor boxes**: Implement 2 anchors per cell with predefined aspect ratios [1:1, 2:1].
   Each anchor predicts independently. Does this improve recall for objects of varying sizes?

5. **Visualise per-cell class distribution**: For each of the 16 cells, compute the most frequent
   assigned class across the training set. Does the distribution look uniform or are certain
   positions biased toward certain classes?

## Key Takeaways

- **Grid-based detection** enables multi-object detection in a single forward pass.
- **GT assignment by centre**: each object is assigned to the grid cell containing its centre.
- **Objectness** gates class and box losses — only positive cells contribute to these losses.
- **NMS** removes duplicate boxes post-inference by suppressing lower-scoring overlapping detections.
- **Grid resolution** is a key hyperparameter: finer grids detect smaller objects but increase
  prediction overhead and training difficulty.
- The limitation of one object per cell is addressed by anchors (YOLOv2+) and later by
  query-based detectors like DETR (Notebook 15).

## Saved Outputs

In [ ]:
save_markdown_report(
    title='Multi-Object Grid Detector — TinyGridDetector',
    summary=(
        f'TinyGridDetector (grid_size={GRID_SIZE}) trained {NUM_EPOCHS} epochs on SyntheticShapes (multi_detection). '
        f'Precision@IoU0.5: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}.'
    ),
    metrics=metrics,
    figures=['grid_detector_training_curve.png', 'grid_detector_predictions.png',
             'grid_detector_nms_comparison.png', 'grid_detector_grid_visualisation.png'],
    tables=[],
    output_path=run_dir / 'grid_detector_report.md',
    device=str(device),
    hyperparams={
        'grid_size': GRID_SIZE, 'num_classes': NUM_CLASSES, 'lambda_box': LAMBDA_BOX,
        'lambda_obj': LAMBDA_OBJ, 'batch_size': BATCH_SIZE, 'epochs': NUM_EPOCHS, 'lr': LR,
    },
    architecture='CNNBackbone → AdaptivePool(4×4) → Conv1x1(n_pred) per cell',
    loss_fn='obj BCE + cls CE + box SmoothL1 (on positive cells only)',
)

update_runs_summary(
    session_name='grid_detector',
    report_path=run_dir / 'grid_detector_report.md',
    metrics=metrics,
    figures=['grid_detector_training_curve.png', 'grid_detector_predictions.png'],
)

print('All outputs saved:')
print(f'  {run_dir}/grid_detector_metrics.json')
print(f'  {run_dir}/grid_detector_training_curve.png')
print(f'  {run_dir}/grid_detector_predictions.png')
print(f'  {run_dir}/grid_detector_report.md')